# Colab Public Face Pretraining

This notebook mounts Google Drive, installs the project dependencies, and runs public face pretraining for the current `EdgeFaceXXS` student model on a large dataset stored on Drive.

It supports the current public dataset layout where:
- `train/` may be sharded into range folders such as `n002_044/`, `n045_086/`, ...
- each shard then contains the actual identity folders such as `n000002/`, `n000003/`, ...
- `val/` may expose identity folders directly

Recommended use:
- Stage A: public face pretraining on Drive dataset
- Save best checkpoint back to Drive
- Later finetune locally or in a second notebook on `clean-core` and `full` internal datasets


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Face-Recognition-Workspace' / 'Attendance_Workspace' / '3_edgeface_training'
DATASET_ROOT = DRIVE_ROOT / 'Face_Recognition'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'

OUTPUT_PREFIX = 'phase3_public_pretrain'
EPOCHS = 20
BATCH_SIZE = 64
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WIDTH_PRESET = 'widened'
RANK_RATIO = 0.7

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)


In [ ]:
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert DATASET_ROOT.exists(), f'Dataset root not found: {DATASET_ROOT}'
assert (DATASET_ROOT / 'train').exists(), f'Missing train split: {DATASET_ROOT / "train"}'
assert (DATASET_ROOT / 'val').exists(), f'Missing val split: {DATASET_ROOT / "val"}'

train_entries = [p for p in (DATASET_ROOT / 'train').iterdir() if p.is_dir()]
val_entries = [p for p in (DATASET_ROOT / 'val').iterdir() if p.is_dir()]
print('train_shard_or_class_count =', len(train_entries))
print('val_class_count =', len(val_entries))
print('sample_train_entries =', [p.name for p in train_entries[:5]])
print('sample_val_entries =', [p.name for p in val_entries[:5]])


In [ ]:
%cd {PROJECT_ROOT}
!pip install -q -r requirements.txt


In [ ]:
import shlex
import subprocess

cmd = [
    'python',
    'scripts/train_phase3.py',
    '--dataset-root', str(DATASET_ROOT),
    '--checkpoints-dir', str(CHECKPOINT_DIR),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--learning-rate', str(LEARNING_RATE),
    '--width-preset', WIDTH_PRESET,
    '--rank-ratio', str(RANK_RATIO),
    '--kd-alpha', '0',
    '--skip-student-bootstrap',
    '--skip-teacher-bootstrap',
    '--output-prefix', OUTPUT_PREFIX,
]

print('Running command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)


## After Training

Expected checkpoint outputs:
- `checkpoints/phase3_public_pretrain_best.pth`
- `checkpoints/phase3_public_pretrain_metrics.json`

Recommended next step after Stage A:
- use the best public-pretrained checkpoint as `--student-weights` for internal `clean-core` finetuning
- then finetune again on the full internal dataset
- evaluate only on the internal held-out split
